<a href="https://colab.research.google.com/github/Elbehery-237/ann-group13/blob/master/src/Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Data Loading, Exploration, and Handling**
---

*DATASET DETAILS HERE*

In [17]:
# Github Pulls

import os, sys, glob

REPO = "ann-group13"
URL = f"https://github.com/Elbehery-237/ann-group13.git"

if not os.path.exists(f"/content/ann-group13"):
    !git clone {URL}
else:
    !git -C /content/ann-group13 pull

sys.path.append(f"/content/ann-group13")

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.06 KiB | 543.00 KiB/s, done.
From https://github.com/Elbehery-237/ann-group13
   f769549..d6ef293  master     -> origin/master
Updating f769549..d6ef293
Fast-forward
 src/models/FourierMLP.ipynb | 1993 +------------------------------------------
 1 file changed, 16 insertions(+), 1977 deletions(-)


In [15]:
# Library Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn

In [14]:
# Loading Feynman Equations Dataset (Medium-level equations)

from huggingface_hub import snapshot_download

SET = "srsd-feynman_medium"
root = snapshot_download(repo_id=f"yoshitomo-matsubara/{SET}", repo_type="dataset", local_dir=f"data/{SET}")
print(sorted(os.listdir(root)))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 165 files:   0%|          | 0/165 [00:00<?, ?it/s]

['.cache', '.gitattributes', 'README.md', 'problem_table.pdf', 'problem_table.png', 'supp_info.json', 'test', 'train', 'true_eq', 'val']


In [16]:
# Creating the CSV with the data

rows = []

for f in sorted(glob.glob(f"{root}/train/*.txt")):
    name = os.path.basename(f).replace(".txt", "")
    tr = np.loadtxt(f)
    va = np.loadtxt(f"{root}/val/{name}.txt")
    te = np.loadtxt(f"{root}/test/{name}.txt")
    X, y = tr[:, :-1], tr[:, -1]
    lo, hi = X.min(0), X.max(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(lo > 0, hi / lo, np.nan)
    rows.append(dict(
        equation=name, d=X.shape[1],
        n_train=len(tr), n_val=len(va), n_test=len(te),
        wide_cols=[i for i, r in enumerate(ratio) if r > 100],
        has_nonpositive=bool((lo <= 0).any()),
        y_min=y.min(), y_max=y.max(),
        n_nan=int(np.isnan(tr).sum())))

inv = pd.DataFrame(rows)
inv.to_csv("data/inventory.csv", index=False)
inv

,equation,d,n_train,n_val,n_test,wide_cols,has_nonpositive,y_min,y_max,n_nan
0,feynman-bonus.18,3,8000,1000,1000,[],True,-1.295998e+13,2.861810e+13,0
1,feynman-bonus.8,2,8000,1000,1000,[],True,1.000693e-24,9.996644e-23,0
2,feynman-i.10.7,2,8000,1000,1000,[1],False,1.001377e-01,1.025202e+01,0
3,feynman-i.11.19,6,8000,1000,1000,[],True,-1.614491e+02,1.232233e+02,0
4,feynman-i.12.11,5,8000,1000,1000,[4],True,-3.687447e+02,4.182652e+02,0
5,feynman-i.12.2,3,8000,1000,1000,[],True,-6.803412e+13,4.227176e+13,0
6,feynman-i.13.12,4,8000,1000,1000,[],False,-2.824805e-09,4.232641e-09,0
7,feynman-i.13.4,4,8000,1000,1000,[],True,3.398820e-03,7.869462e+02,0
8,feynman-i.15.10,2,8000,1000,1000,[],True,-9.541719e+06,9.691889e+06,0
9,feynman-i.16.6,2,8000,1000,1000,[],True,-1.730135e+08,1.703527e+08,0
